<a href="https://colab.research.google.com/github/Saadd-x/FYP-Work/blob/main/Drone%20Dataset%20YOLOv5n%20Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FYP Drone Detection — YOLOv5n Training

Clean notebook for training YOLOv5n on the Seraphim drone dataset.

This notebook will:
1. Mount Google Drive
2. Check GPU
3. Download the Seraphim dataset
4. Prepare the same one-class YOLO dataset with a reproducible 90/10 split
5. Save the prepared dataset as `drone_dataset.zip` to Drive BEFORE training
6. Clone YOLOv5
7. Train YOLOv5n for 30 epochs
8. Save `best.pt` and `last.pt` to Drive
9. Provide resume and video-testing cells

Your previous YOLOv5s run remains separate in `FYP_Drone/runs/test_drone/`.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
FYP_DIR = Path('/content/drive/MyDrive/FYP_Drone')
FYP_DIR.mkdir(parents=True, exist_ok=True)
print("FYP folder:", FYP_DIR)

Mounted at /content/drive
FYP folder: /content/drive/MyDrive/FYP_Drone


## 2. Check GPU

Before running this cell: **Runtime → Change runtime type → GPU**.

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Go to Runtime → Change runtime type → GPU.")

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Download the Seraphim dataset

In [4]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="lgrzybowski/seraphim-drone-detection-dataset",
    repo_type="dataset",
    local_dir="/content/seraphim"
)

print("Dataset downloaded.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Dataset downloaded.


## 4. Extract ZIP files if the dataset contains archives

In [12]:
import zipfile
from pathlib import Path

dataset_path = Path("/content/seraphim")

for zip_path in dataset_path.rglob("*.zip"):
    print("Extracting:", zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(zip_path.parent)

print("Extraction complete.")

Extracting: /content/seraphim/train/labels/batch_001.zip
Extracting: /content/seraphim/train/images/batch_001.zip
Extracting: /content/seraphim/train/images/batch_003.zip
Extracting: /content/seraphim/train/images/batch_004.zip
Extracting: /content/seraphim/train/images/batch_002.zip
Extracting: /content/seraphim/test/labels/batch_001.zip
Extracting: /content/seraphim/test/images/batch_001.zip
Extraction complete.


## 5. Prepare the YOLO dataset

This uses one class (`drone`) and a fixed 90/10 train/validation split. Files are sorted before shuffling so the split is reproducible.

In [13]:
import random
import shutil
from pathlib import Path

random.seed(42)

SOURCE = Path("/content/seraphim")
BASE = Path("/content/drone_dataset")

for folder in [
    BASE / "images/train",
    BASE / "images/val",
    BASE / "labels/train",
    BASE / "labels/val",
]:
    folder.mkdir(parents=True, exist_ok=True)

source_images = SOURCE / "train/images"
source_labels = SOURCE / "train/labels"

images = [
    img for img in source_images.glob("*")
    if (source_labels / f"{img.stem}.txt").exists()
]

random.shuffle(images)

split = int(len(images) * 0.90)

train_images = images[:split]
val_images = images[split:]

print("Total:", len(images))
print("Train:", len(train_images))
print("Validation:", len(val_images))

for img in train_images:
    shutil.copy2(img, BASE / "images/train" / img.name)
    shutil.copy2(
        source_labels / f"{img.stem}.txt",
        BASE / "labels/train" / f"{img.stem}.txt"
    )

for img in val_images:
    shutil.copy2(img, BASE / "images/val" / img.name)
    shutil.copy2(
        source_labels / f"{img.stem}.txt",
        BASE / "labels/val" / f"{img.stem}.txt"
    )

print("Dataset prepared.")

Total: 75134
Train: 67620
Validation: 7514
Dataset prepared.


## 6. Create `data.yaml`

In [14]:
data_yaml = """
path: /content/drone_dataset

train: images/train
val: images/val

nc: 1
names:
  0: drone
"""

with open("/content/drone_dataset/data.yaml", "w") as f:
    f.write(data_yaml.strip())

print("data.yaml created.")

data.yaml created.


## 7. Verify the dataset

In [16]:
from pathlib import Path

OUT = Path("/content/drone_dataset")
DATA_YAML = OUT / "data.yaml"

train_count = len(list((OUT / "images" / "train").glob("*")))
val_count = len(list((OUT / "images" / "val").glob("*")))

print("Train images:", train_count)
print("Validation images:", val_count)
print("data.yaml exists:", DATA_YAML.exists())

assert train_count > 0, "No training images found!"
assert val_count > 0, "No validation images found!"
assert DATA_YAML.exists(), "data.yaml not found!"

print("Dataset verification passed.")

Train images: 67620
Validation images: 7514
data.yaml exists: True
Dataset verification passed.


## 8. IMPORTANT — Save the prepared dataset to Drive

Do this BEFORE the long training. After this, future Colab runtimes can restore the dataset from Drive without downloading Seraphim again.

In [ ]:
import shutil
from pathlib import Path

zip_base = FYP_DIR / "drone_dataset"
zip_file = Path(str(zip_base) + ".zip")

if zip_file.exists():
    zip_file.unlink()

print("Creating ZIP. This may take a while...")
created = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir="/content",
    base_dir="drone_dataset"
)

size_gb = Path(created).stat().st_size / (1024**3)
print("Saved:", created)
print("ZIP size (GB):", round(size_gb, 2))

Creating ZIP. This may take a while...


## 9. Clone YOLOv5 and install requirements

In [ ]:
%cd /content

!rm -rf /content/yolov5
!git clone https://github.com/ultralytics/yolov5.git

%cd /content/yolov5
!pip install -q -r requirements.txt

print("YOLOv5 ready.")

## 10. Train YOLOv5n for 30 epochs

This is a NEW experiment. It does not use the YOLOv5s checkpoint.

In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 30 \
    --data /content/drone_dataset/data.yaml \
    --weights yolov5n.pt \
    --name drone_v5n \
    --project /content/drive/MyDrive/FYP_Drone/runs \
    --save-period 1

## 11. Check the saved YOLOv5n files

In [ ]:
from pathlib import Path

RUN_DIR = Path("/content/drive/MyDrive/FYP_Drone/runs/drone_v5n")

print("Run folder:", RUN_DIR)
print("Exists:", RUN_DIR.exists())

if RUN_DIR.exists():
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file():
            print(p)

## 12. If Colab disconnects before training finishes

Do NOT restart from zero. Restore the saved dataset ZIP and resume from `last.pt`.

In [ ]:
# Restore the prepared dataset after a runtime reset:
!rm -rf /content/drone_dataset
!unzip -q "/content/drive/MyDrive/FYP_Drone/drone_dataset.zip" -d /content/

# After cloning/installing YOLOv5 again:
%cd /content/yolov5
!python train.py \
    --resume /content/drive/MyDrive/FYP_Drone/runs/drone_v5n/weights/last.pt

## 13. Test the best YOLOv5n model on a drone video

In [ ]:
%cd /content/yolov5

!python detect.py \
    --weights /content/drive/MyDrive/FYP_Drone/runs/drone_v5n/weights/best.pt \
    --source "/content/your_video.mp4" \
    --img 640 \
    --conf 0.25

## 14. Final FYP comparison

Keep the existing YOLOv5s experiment as the baseline:

`FYP_Drone/runs/test_drone/`

New YOLOv5n experiment:

`FYP_Drone/runs/drone_v5n/`

Compare:
- Precision
- Recall
- mAP@0.5
- mAP@0.5:0.95
- Model size
- FPS/latency
- FPGA resource utilization

Do not delete the YOLOv5s baseline.